# Safety Jacket Detection - Auto-Annotation & Training

This notebook is designed to run on a dedicated GPU (like Google Colab T4, Kaggle, or a local GPU machine).

It will:
1. Download your unannotated image dataset from HuggingFace.
2. Automatically annotate the images with bounding boxes using **Grounding DINO** (Zero-Shot Object Detection).
3. Train a **YOLOv8** model on the newly annotated dataset.
4. Export the trained weights for you to download.

In [ ]:
!pip install datasets transformers ultralytics torch torchvision opencv-python supervision
!pip install -q git+https://github.com/IDEA-Research/GroundingDINO.git

## 1. Download Dataset

In [ ]:
import os
from datasets import load_dataset
from PIL import Image

dataset_name = "UCUResearchLab/Reflector-Jackets"
print(f"Downloading {dataset_name}...")
ds = load_dataset(dataset_name)

os.makedirs("dataset/images/train", exist_ok=True)
os.makedirs("dataset/labels/train", exist_ok=True)

images_dir = "dataset/images/train"

count = 0
for split in ds.keys():
    for item in ds[split]:
        # Handle both forms of image storing in hf datasets
        if 'image' in item:
            image = item['image']
        elif 'img' in item:
            image = item['img']
        else:
            # Find the first key that contains a PIL Image
            image_key = next((k for k, v in item.items() if hasattr(v, 'mode') and hasattr(v, 'save')), None)
            if image_key:
                image = item[image_key]
            else:
                continue

        image_path = os.path.join(images_dir, f"img_{count}.jpg")
        if image.mode != "RGB":
            image = image.convert("RGB")
        image.save(image_path)
        count += 1

print(f"Successfully downloaded and saved {count} images.")

## 2. Auto-Annotate with Grounding DINO
We use Grounding DINO via Hugging Face `transformers` to find the safety jackets.

In [ ]:
import cv2
import torch
from transformers import pipeline
import glob

print("Loading Grounding DINO...")
device = "cuda" if torch.cuda.is_available() else "cpu"
detector = pipeline(task="zero-shot-object-detection", model="IDEA-Research/grounding-dino-tiny", device=device)

candidate_labels = ["safety jacket", "high visibility vest", "reflector jacket"]
CONFIDENCE_THRESHOLD = 0.3

labels_dir = "dataset/labels/train"
image_paths = glob.glob(os.path.join(images_dir, "*.jpg"))

print(f"Annotating {len(image_paths)} images...")

for image_path in image_paths:
    image = Image.open(image_path).convert("RGB")
    results = detector(image, candidate_labels=candidate_labels)
    
    width, height = image.size
    base_name = os.path.basename(image_path).replace(".jpg", ".txt")
    label_path = os.path.join(labels_dir, base_name)

    with open(label_path, "w") as f:
        for result in results:
            score = result["score"]
            if score < CONFIDENCE_THRESHOLD:
                continue
            
            # YOLO format: class x_center y_center width height (normalized 0-1)
            box = result["box"]
            xmin, ymin, xmax, ymax = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
            
            x_center = ((xmin + xmax) / 2) / width
            y_center = ((ymin + ymax) / 2) / height
            box_width = (xmax - xmin) / width
            box_height = (ymax - ymin) / height
            
            class_id = 0 # 0 for safety jacket
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}\n")
            
print("Annotation complete!")

## 3. Train YOLOv8
First, create the YAML file required by YOLO. Then, start training.

In [ ]:
yaml_content = """
path: ./dataset
train: images/train
val: images/train  # Using train as val for simplicity if we don't have separate splits

names:
  0: safety_jacket
"""

with open("dataset.yaml", "w") as f:
    f.write(yaml_content)

from ultralytics import YOLO

# Load a pretrained YOLOv8 nano model (fastest)
model = YOLO("yolov8n.pt")

# Train the model
print("Starting YOLOv8 training...")
results = model.train(
    data="dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=device
)

## 4. Export & Download Model Weights
The best model weights are saved in `runs/detect/train/weights/best.pt`.

In [ ]:
import os

best_model_path = "runs/detect/train/weights/best.pt"
if os.path.exists(best_model_path):
    print(f"Model successfully trained! The weights are saved at {os.path.abspath(best_model_path)}.")
    
    try:
        from google.colab import files
        print("Downloading best.pt to your local machine...")
        files.download(best_model_path)
    except ImportError:
        print("Not running in Google Colab. Please download the file manually if needed.")
else:
    print(f"Could not find {best_model_path}. Please check the runs/ directory manually.")